# [TEST] Evaluate O*NET Tag Accuracy — Capital One

Capital One 데이터로 step-4 정확도 평가 파이프라인 테스트.
Spark 도입 전 처리 시간 및 결과 품질 확인용.

**Model:** Claude Sonnet 4.5 (`claude-sonnet-4-5-20250929`)

**Verdict scale:**
- `exact` — onet_tag가 해당 역할과 직접 일치
- `broader` — onet_tag가 해당 역할을 포함하는 상위 카테고리
- `mismatch` — onet_tag가 맞지 않음

**Scope:** Capital One (183 titles, 10개씩 배치, JD 전체 포함)

In [11]:
import pandas as pd
import subprocess
import json
import re
import time
import os
from pathlib import Path

INPUT_FILE = Path('../step-3-add-descriptions/top10_onet_tagged_with_desc/Capital One_verified.csv')
OUTPUT_DIR = Path('capital_one_eval')
OUTPUT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 20

In [12]:
# Load Capital One data
df = pd.read_csv(INPUT_FILE)
print(f'Capital One: {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'Batches: {(len(df) - 1) // BATCH_SIZE + 1}')
df.head(3)

Capital One: 183 rows
Columns: ['onet_tag', 'company_name', 'raw_title', 'description']
Batches: 10


,onet_tag,company_name,raw_title,description
0,"Fraud Examiners, Investigators and Analysts",Capital One,Anti-Money Laundering (AML) Sr. Investigator I...,"West Creek 3 (12073), United States of America..."
1,"Fraud Examiners, Investigators and Analysts",Capital One,Anti-Money Laundering Subject Matter Expert (S...,"West Creek 3 (12073), United States of America..."
2,"Fraud Examiners, Investigators and Analysts",Capital One,"Principal Associate, CB.xd - Fraud & Disputes","Locations: VA - Richmond, United States of Ame..."


In [13]:
def evaluate_batch(batch: list[dict]) -> tuple[list[dict], dict]:
    """
    Ask Claude Sonnet 4.5 to evaluate a batch of (raw_title, description, onet_tag).

    Returns:
        verdicts: list of {raw_title, verdict, reason}
        usage:    {input_tokens, output_tokens, cost_usd}
    """
    # Number each item for stable mapping
    numbered = [
        f"{i+1}. raw_title: {item['raw_title']}\n"
        f"   onet_tag: {item['onet_tag']}\n"
        f"   description: {item['description']}"
        for i, item in enumerate(batch)
    ]
    items_text = '\n\n'.join(numbered)

    prompt = f"""You are evaluating whether an O*NET job category is a reasonable match for a LinkedIn job posting.

For each item, evaluate if the onet_tag fits the raw_title and job description.
Verdict options:
- exact: onet_tag directly and specifically matches this role
- broader: onet_tag is a valid broader/umbrella category that includes this role
- mismatch: onet_tag does not fit this job at all

Jobs to evaluate:
{items_text}

Respond with one line per item in this exact format:
<number>|<verdict>|<reason>

Example:
1|exact|Direct match for software engineering role
2|broader|Data scientists is a broader umbrella for analytics work
3|mismatch|Operations management does not fit this technical role

Rules:
- One line per item, no extra lines
- Use | as delimiter
- reason: one plain sentence, no special characters"""

    with open('temp_eval_prompt.txt', 'w', encoding='utf-8') as f:
        f.write(prompt)

    result = subprocess.run(
        'type temp_eval_prompt.txt | claude --print --output-format json --model sonnet -',
        capture_output=True, text=True, shell=True
    )

    # Parse CLI JSON envelope
    try:
        cli_json = json.loads(result.stdout)
        response_text = cli_json.get('result', '').strip()
        u = cli_json.get('usage', {})
        usage = {
            'input_tokens': u.get('input_tokens', 0)
                          + u.get('cache_creation_input_tokens', 0)
                          + u.get('cache_read_input_tokens', 0),
            'output_tokens': u.get('output_tokens', 0),
            'cost_usd': cli_json.get('total_cost_usd', 0.0)
        }
    except json.JSONDecodeError:
        response_text = result.stdout.strip()
        usage = {'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0.0}

    # Parse line-based response: number|verdict|reason
    verdicts = []
    for line in response_text.split('\n'):
        line = line.strip()
        if not line:
            continue
        parts = line.split('|', maxsplit=2)
        if len(parts) == 3:
            try:
                idx = int(parts[0]) - 1
                verdicts.append({
                    'raw_title': batch[idx]['raw_title'],
                    'verdict': parts[1].strip(),
                    'reason': parts[2].strip()
                })
            except (ValueError, IndexError):
                print(f'    Warning: Could not parse line: {line}')
        else:
            print(f'    Warning: Unexpected line format: {line}')

    if len(verdicts) != len(batch):
        print(f'    Warning: Expected {len(batch)} verdicts, got {len(verdicts)}')

    return verdicts, usage

In [14]:
import datetime

output_file = OUTPUT_DIR / 'capital_one_eval.csv'
eval_rows = []
total = len(df)

total_input_tokens = 0
total_output_tokens = 0
total_cost_usd = 0.0

start_time = datetime.datetime.now()
print(f'Start: {start_time.strftime("%H:%M:%S")}')
print(f'Processing Capital One ({total} rows, {(total-1)//BATCH_SIZE+1} batches)...\n')

for i in range(0, total, BATCH_SIZE):
    batch_df = df.iloc[i:i+BATCH_SIZE]

    batch = []
    for _, row in batch_df.iterrows():
        desc = str(row['description']) if pd.notna(row['description']) else ''
        batch.append({
            'raw_title': row['raw_title'],
            'description': desc,
            'onet_tag': row['onet_tag']
        })

    verdicts, usage = evaluate_batch(batch)

    total_input_tokens += usage['input_tokens']
    total_output_tokens += usage['output_tokens']
    total_cost_usd += usage['cost_usd']

    print(f'  Batch {i//BATCH_SIZE + 1}/{(total-1)//BATCH_SIZE + 1}: '
          f'{i+1}-{min(i+BATCH_SIZE, total)}/{total} | '
          f'tokens={usage["input_tokens"]}+{usage["output_tokens"]} | '
          f'cost=${usage["cost_usd"]:.4f}')

    verdict_map = {v['raw_title']: v for v in verdicts}
    for _, row in batch_df.iterrows():
        v = verdict_map.get(row['raw_title'], {})
        eval_rows.append({
            'onet_tag': row['onet_tag'],
            'company_name': row['company_name'],
            'raw_title': row['raw_title'],
            'verdict': v.get('verdict', 'error'),
            'reason': v.get('reason', '')
        })

    time.sleep(1)

end_time = datetime.datetime.now()
elapsed = end_time - start_time

df_eval = pd.DataFrame(eval_rows)
df_eval.to_csv(output_file, index=False, encoding='utf-8-sig')

if os.path.exists('temp_eval_prompt.txt'):
    os.remove('temp_eval_prompt.txt')

print(f'\n{"="*50}')
print(f'End     : {end_time.strftime("%H:%M:%S")}')
print(f'Elapsed : {elapsed}')
print(f'Tokens  : {total_input_tokens:,} input / {total_output_tokens:,} output')
print(f'Cost    : ${total_cost_usd:.4f} USD')
print(f'Saved to: {output_file}')

Start: 23:25:05
Processing Capital One (183 rows, 10 batches)...

  Batch 1/10: 1-20/183 | tokens=48252+468 | cost=$0.2107
  Batch 2/10: 21-40/183 | tokens=49688+534 | cost=$0.2213
  Batch 3/10: 41-60/183 | tokens=50806+600 | cost=$0.2300
  Batch 4/10: 61-80/183 | tokens=49216+603 | cost=$0.2201
  Batch 5/10: 81-100/183 | tokens=47079+506 | cost=$0.2043
  Batch 6/10: 101-120/183 | tokens=45552+531 | cost=$0.1954
  Batch 7/10: 121-140/183 | tokens=49842+545 | cost=$0.2226
  Batch 8/10: 141-160/183 | tokens=50524+548 | cost=$0.2269
  Batch 9/10: 161-180/183 | tokens=51185+595 | cost=$0.2322
  Batch 10/10: 181-183/183 | tokens=26400+101 | cost=$0.0650

End     : 23:27:34
Elapsed : 0:02:29.102822
Tokens  : 468,544 input / 5,031 output
Cost    : $2.0285 USD
Saved to: capital_one_eval\capital_one_eval.csv


In [17]:
# Results summary
total = len(df_eval)
counts = df_eval['verdict'].value_counts()

print(f'=== Capital One Accuracy ===')
print(f'Total titles : {total}')
print(f'Elapsed      : {elapsed}')
print(f'Tokens       : {total_input_tokens:,} input / {total_output_tokens:,} output')
print(f'Cost         : ${total_cost_usd:.4f} USD')
print()
for verdict in ['exact', 'broader', 'mismatch', 'error']:
    n = counts.get(verdict, 0)
    print(f'  {verdict:<10}: {n:>3} ({n/total*100:.1f}%)')

valid = counts.get('exact', 0) + counts.get('broader', 0)
print(f'\n  Valid (exact + broader): {valid} ({valid/total*100:.1f}%)')

=== Capital One Accuracy ===
Total titles : 183
Elapsed      : 0:02:29.102822
Tokens       : 468,544 input / 5,031 output
Cost         : $2.0285 USD

  exact     :  95 (51.9%)
  broader   :  59 (32.2%)
  mismatch  :  29 (15.8%)
  error     :   0 (0.0%)

  Valid (exact + broader): 154 (84.2%)


In [18]:
# Sample mismatches for review
print('=== Sample Mismatches ===')
mismatches = df_eval[df_eval['verdict'] == 'mismatch'][['raw_title', 'onet_tag', 'reason']]
print(mismatches.head(20).to_string(index=False))

=== Sample Mismatches ===
                                                                                          raw_title                                                     onet_tag                                                                                                                                                                                          reason
                                                      Principal Associate, CB.xd - Fraud & Disputes                  Fraud Examiners, Investigators and Analysts                                                                                                                This is a UX content designer role not a fraud examiner or investigator position
                                                   Senior Associate, Product Management, Card Fraud                  Fraud Examiners, Investigators and Analysts                                                                                     This is a product manag

In [19]:
# Sample exact matches for reference
print('=== Sample Exact Matches ===')
exacts = df_eval[df_eval['verdict'] == 'exact'][['raw_title', 'onet_tag', 'reason']]
print(exacts.head(10).to_string(index=False))

=== Sample Exact Matches ===
                                                                                     raw_title                                                     onet_tag                                                                                                                    reason
Anti-Money Laundering (AML) Sr. Investigator III, Transaction Monitoring Operations Team (TMO)                  Fraud Examiners, Investigators and Analysts                                         AML investigator role directly matches fraud examiners and investigators category
         Anti-Money Laundering Subject Matter Expert (SME) - Special Investigations Unit (SIU)                  Fraud Examiners, Investigators and Analysts                          AML special investigations unit SME role is a direct match for fraud examiners and investigators
            Relationship Manager, Business Cards & Payments - Acquisitions (Southeast Florida) Securities, Commodities, and Financial Ser